# COCO segmentation metadata (pixelwise + polygon)

This notebook shows how to:
- Read COCO segmentation metadata as **pixelwise masks** and use the metadata APIs:
  - `get_pixelwise_labels()`
  - `get_random_mask_pixel()`
  - `get_random_object_bbox()`
- Read COCO segmentation metadata as **polygon masks** and use:
  - `get_mask_count()`
  - `get_mask_coordinates()`
  - `get_select_mask()`


In [ ]:
import os
import ctypes

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from amd.rocal.pipeline import Pipeline
import amd.rocal.fn as fn
import amd.rocal.types as types


In [ ]:
def _tensorlist_to_torch(tl, output_memory_type):
    """Copy rocAL output tensorlist to a torch tensor."""
    dims = tl.dimensions()
    torch_dtype = getattr(torch, tl.dtype())
    out = torch.empty(dims, dtype=torch_dtype)
    tl.copy_data(ctypes.c_void_p(out.data_ptr()), output_memory_type)
    return out


def _crop_valid(img_hwc, w, h):
    # img_hwc: HxWxC
    h = int(min(h, img_hwc.shape[0]))
    w = int(min(w, img_hwc.shape[1]))
    return img_hwc[:h, :w]


def _reshape_mask(mask, w, h):
    mask = np.asarray(mask)
    if mask.ndim == 2:
        if mask.shape == (h, w):
            return mask
        if mask.shape == (w, h):
            return mask.T
    if mask.size == w * h:
        return mask.reshape((h, w))
    raise ValueError(f"Unexpected mask shape {mask.shape} for (w,h)=({w},{h})")


def _show_pixelwise_sample(img_hwc_u8, mask_hw, random_pixel, random_bbox, title=None):
    fig, ax = plt.subplots(dpi=140)
    ax.imshow(img_hwc_u8)
    ax.imshow(mask_hw, cmap="jet", alpha=0.35)

    row, col = int(random_pixel[0]), int(random_pixel[1])
    ax.scatter([col], [row], s=20, c="#ff0000")

    y0, x0, y1, x1 = [int(v) for v in random_bbox]  # box format (y0,x0,y1,x1)
    rect = patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=2, edgecolor="#00b7ff", facecolor="none")
    ax.add_patch(rect)

    ax.axis("off")
    if title:
        ax.set_title(title)
    plt.show()


<div class="alert alert-block alert-warning">
<b>Note:</b> Set the ROCAL_DATA_PATH environment variable before running the notebook.
</div>


In [ ]:
rocal_data_path = os.environ.get("ROCAL_DATA_PATH")
if rocal_data_path is None:
    raise EnvironmentError("ROCAL_DATA_PATH environment variable is not set. Please set it to the correct path.")
else:
    print(f"ROCAL_DATA_PATH IS SET TO: {rocal_data_path}")

image_path = os.path.join(
    rocal_data_path,
    "rocal_data/coco/coco_10_img_keypoints/person_keypoints_10images_val2017/",
)
annotation_path = os.path.join(
    rocal_data_path,
    "rocal_data/coco/coco_10_img_keypoints/annotations/person_keypoints_val2017.json",
)

assert os.path.exists(image_path), f"Missing image_path: {image_path}"
assert os.path.exists(annotation_path), f"Missing annotation_path: {annotation_path}"

batch_size = 2
num_threads = 1
device_id = 0
seed = 0

# Decoder output is padded up to these values; crop to per-sample (w,h) for visualization.
max_decoded_width = 640
max_decoded_height = 640

pipe = Pipeline(batch_size=batch_size, num_threads=num_threads, device_id=device_id, seed=seed, rocal_cpu=True)
with pipe:
    # Pixelwise-mask mode
    # Configure random_mask_pixel to select from foreground pixels (value > 0).
    meta, bboxes, labels = fn.readers.coco(
        annotations_file=annotation_path,
        pixelwise_masks=True,
        is_foreground=True,
        value=0,
        is_threshold=True,
    )
    images = fn.decoders.image(
        meta,
        file_root=image_path,
        annotations_file=annotation_path,
        output_type=types.RGB,
        random_shuffle=False,
        max_decoded_width=max_decoded_width,
        max_decoded_height=max_decoded_height,
    )
    pipe.set_outputs(images)
pipe.build()

# Run one batch and visualize samples
img_sizes = np.zeros((batch_size * 2), dtype=np.int32)
pipe.rocal_run()
pipe.get_img_sizes(img_sizes)

out = _tensorlist_to_torch(pipe.get_output_tensors()[0], pipe._output_memory_type)  # NHWC uint8 by default
pixelwise = pipe.get_pixelwise_labels()
random_pixels = pipe.get_random_mask_pixel()
random_bboxes = pipe.get_random_object_bbox("box", k_largest=-1, foreground_prob=1.0, cache_objects=False)

for bi in range(batch_size):
    w = int(img_sizes[bi * 2 + 0])
    h = int(img_sizes[bi * 2 + 1])

    img_hwc = out[bi].cpu().numpy()
    img_hwc = _crop_valid(img_hwc, w, h)
    img_hwc_u8 = img_hwc.astype(np.uint8)

    mask_hw = _reshape_mask(pixelwise[bi], w, h)
    rp = np.asarray(random_pixels[bi], dtype=np.int32)
    rb = np.asarray(random_bboxes[bi], dtype=np.int32)

    _show_pixelwise_sample(img_hwc_u8, mask_hw, rp, rb, title=f"Pixelwise mask + random mask pixel (sample {bi})")

In [ ]:
# polygon-mask mode + select_masks

pipe_poly = Pipeline(batch_size=batch_size, num_threads=num_threads, device_id=device_id, seed=seed, rocal_cpu=True)
with pipe_poly:
    meta, bboxes, labels = fn.readers.coco(
        annotations_file=annotation_path,
        polygon_masks=True,
    )
    images = fn.decoders.image(
        meta,
        file_root=image_path,
        annotations_file=annotation_path,
        output_type=types.RGB,
        random_shuffle=False,
        max_decoded_width=max_decoded_width,
        max_decoded_height=max_decoded_height,
    )
    pipe_poly.set_outputs(images)
pipe_poly.build()

img_sizes = np.zeros((batch_size * 2), dtype=np.int32)
pipe_poly.rocal_run()
pipe_poly.get_img_sizes(img_sizes)

out = _tensorlist_to_torch(pipe_poly.get_output_tensors()[0], pipe_poly._output_memory_type)
bbox_total = pipe_poly.get_bounding_box_count()
mask_count = np.zeros(bbox_total, dtype=np.int32)
mask_total = pipe_poly.get_mask_count(mask_count)
polygon_size = np.zeros(mask_total, dtype=np.int32)
polygons = pipe_poly.get_mask_coordinates(polygon_size, mask_count)
selected = pipe_poly.get_select_mask([0])  # pick polygon id 0 per-image

# Visualize selected polygons for sample 0
bi = 0
w = int(img_sizes[bi * 2 + 0])
h = int(img_sizes[bi * 2 + 1])
img_hwc = _crop_valid(out[bi].cpu().numpy(), w, h).astype(np.uint8)

fig, ax = plt.subplots(dpi=140)
ax.imshow(img_hwc)

sel = selected[bi] if isinstance(selected, list) else {}
for _, polys in sel.items():
    for coords in polys:
        pts = np.asarray(coords, dtype=np.float32).reshape((-1, 2))
        ax.plot(pts[:, 0], pts[:, 1], color="#00ff00", linewidth=2)

ax.axis("off")
ax.set_title("Polygon select_mask overlay (sample 0)")
plt.show()
